In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
import json
import torch

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print(f'PyTorch: {torch.__version__} | Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

PyTorch: 2.10.0+cu130 | Device: CUDA


In [2]:
from datasets import load_dataset

print('Chargement du corpus...')
corpus_ds   = load_dataset('vidore/vidore_v3_pharmaceuticals', 'corpus',   split='test')
queries_ds  = load_dataset('vidore/vidore_v3_pharmaceuticals', 'queries',  split='test')
qrels_ds    = load_dataset('vidore/vidore_v3_pharmaceuticals', 'qrels',    split='test')

# Conversion en DataFrames
df_corpus  = corpus_ds.to_pandas()
df_queries = queries_ds.to_pandas()
df_qrels   = qrels_ds.to_pandas()

print('\nColonnes corpus  :', df_corpus.columns.tolist())
print('Colonnes queries :', df_queries.columns.tolist())
print('Colonnes qrels   :', df_qrels.columns.tolist())

Chargement du corpus...

Colonnes corpus  : ['corpus_id', 'image', 'doc_id', 'markdown', 'page_number_in_doc']
Colonnes queries : ['query_id', 'query', 'language', 'query_types', 'query_format', 'content_type', 'raw_answers', 'query_generator', 'query_generation_pipeline', 'source_type', 'query_type_for_generation', 'answer']
Colonnes qrels   : ['query_id', 'corpus_id', 'score', 'content_type', 'bounding_boxes']


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# BioBERT fine-tuné sur STS biomédicale — adapté au vocabulaire pharmaceutique/clinique
# Dimension : 768 | CPU-friendly
MODEL_NAME_A = 'pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb'
print(f'Chargement modèle : {MODEL_NAME_A}')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_a = SentenceTransformer(MODEL_NAME_A, device='cpu')
print('Modèle chargé.')

Chargement modèle : pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé.


In [4]:
# Préparation du corpus textuel
# On préfixe avec 'passage: ' selon les recommandations multilingual-e5
corpus_texts = [
    f"passage: {row['markdown']}" if row['markdown'] else 'passage: [empty page]'
    for _, row in df_corpus.iterrows()
]
corpus_ids = df_corpus['corpus_id'].tolist()

print(f'Encodage de {len(corpus_texts):,} pages...')
BATCH_SIZE = 64
corpus_embeddings_a = model_a.encode(
    corpus_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f'Embeddings corpus shape: {corpus_embeddings_a.shape}')

Encodage de 2,313 pages...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Embeddings corpus shape: (2313, 768)


In [5]:
# Encodage des requêtes
query_texts = [
    f"query: {row['query']}" for _, row in df_queries.iterrows()
]
query_ids = df_queries['query_id'].tolist()

print(f'Encodage de {len(query_texts):,} requêtes...')
query_embeddings_a = model_a.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f'Embeddings requêtes shape: {query_embeddings_a.shape}')

Encodage de 2,184 requêtes...


Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Embeddings requêtes shape: (2184, 768)


In [6]:
def compute_retrieval_results(query_embeddings, corpus_embeddings, query_ids, corpus_ids, k=10):
    """
    Calcule les top-k résultats de retrieval pour chaque requête.
    Retourne un dict {query_id: [(corpus_id, score), ...]}
    """
    results = {}
    # Calcul de la similarité cosinus par batch pour économiser la mémoire
    batch_size = 256
    for i in tqdm(range(0, len(query_ids), batch_size), desc='Computing similarities'):
        batch_q_emb = query_embeddings[i:i+batch_size]
        batch_q_ids = query_ids[i:i+batch_size]
        sims = cosine_similarity(batch_q_emb, corpus_embeddings)  # (batch, corpus)
        top_k_indices = np.argsort(-sims, axis=1)[:, :k]
        for j, qid in enumerate(batch_q_ids):
            results[qid] = [
                (corpus_ids[idx], float(sims[j, idx]))
                for idx in top_k_indices[j]
            ]
    return results

print('Calcul des résultats de retrieval (Approche A)...')
results_a = compute_retrieval_results(
    query_embeddings_a, corpus_embeddings_a, query_ids, corpus_ids, k=10
)
print(f'Résultats calculés pour {len(results_a):,} requêtes.')

Calcul des résultats de retrieval (Approche A)...


Computing similarities:   0%|          | 0/9 [00:00<?, ?it/s]

Résultats calculés pour 2,184 requêtes.


In [7]:
def compute_ndcg_at_k(results, df_qrels, k=5):
    """
    Calcule le nDCG@k pour chaque requête.
    results: dict {query_id: [(corpus_id, score), ...]}
    df_qrels: DataFrame avec colonnes query_id, corpus_id, score
    """
    # Construire le dict de ground truth {query_id: {corpus_id: score}}
    gt = defaultdict(dict)
    for _, row in df_qrels.iterrows():
        gt[row['query_id']][row['corpus_id']] = row['score']

    ndcg_scores = {}
    for qid, ranked_list in results.items():
        if qid not in gt:
            continue
        relevant = gt[qid]
        # DCG@k
        dcg = 0.0
        for rank, (cid, _) in enumerate(ranked_list[:k], start=1):
            rel = relevant.get(cid, 0)
            dcg += (2**rel - 1) / np.log2(rank + 1)
        # IDCG@k (tri par score décroissant)
        ideal_rels = sorted(relevant.values(), reverse=True)[:k]
        idcg = sum(
            (2**rel - 1) / np.log2(rank + 1)
            for rank, rel in enumerate(ideal_rels, start=1)
        )
        ndcg_scores[qid] = dcg / idcg if idcg > 0 else 0.0
    return ndcg_scores

ndcg_a = compute_ndcg_at_k(results_a, df_qrels, k=5)
mean_ndcg_a = np.mean(list(ndcg_a.values()))
print(f'\n{"="*50}')
print(f'  Approche A — nDCG@5 : {mean_ndcg_a:.4f}')
print(f'  Médiane             : {np.median(list(ndcg_a.values())):.4f}')
print(f'  % requêtes à 0.0   : {sum(1 for v in ndcg_a.values() if v == 0) / len(ndcg_a) * 100:.1f}%')
print(f'  % requêtes à 1.0   : {sum(1 for v in ndcg_a.values() if v == 1) / len(ndcg_a) * 100:.1f}%')
print(f'  N requêtes évaluées: {len(ndcg_a):,}')
print('='*50)


  Approche A — nDCG@5 : 0.1029
  Médiane             : 0.0000
  % requêtes à 0.0   : 78.5%
  % requêtes à 1.0   : 2.7%
  N requêtes évaluées: 2,184


In [10]:
df_corpus['markdown_len'] = df_corpus['markdown'].fillna('').str.len()

# --- Analyse qualitative des 10 pires résultats ---
worst_queries_a = sorted(ndcg_a.items(), key=lambda x: x[1])[:10]

print('TOP 10 PIRES REQUÊTES — Approche A (textuelle)')
print('='*80)

gt_dict = defaultdict(dict)
for _, row in df_qrels.iterrows():
    gt_dict[row['query_id']][row['corpus_id']] = row['score']

query_text_dict  = dict(zip(df_queries['query_id'], df_queries['query']))
corpus_meta_dict = dict(zip(df_corpus['corpus_id'],
                            zip(df_corpus['doc_id'],
                                df_corpus['page_number_in_doc'],
                                df_corpus['markdown_len'])))

worst_analysis = []
for qid, score in worst_queries_a:
    query_text = query_text_dict.get(qid, 'N/A')
    relevant_pages = gt_dict.get(qid, {})
    retrieved_top1 = results_a[qid][0][0] if results_a[qid] else None

    # Info sur les pages pertinentes attendues
    rel_info = []
    for cid, rel_score in relevant_pages.items():
        meta = corpus_meta_dict.get(cid, ('?', '?', 0))
        rel_info.append(f'{meta[0]} p.{meta[1]} (score={rel_score}, ocr_len={meta[2]})')

    print(f'\nnDCG@5={score:.3f} | QID: {qid}')
    print(f'  Requête   : {query_text}')
    print(f'  Attendu   : {" | ".join(rel_info)}')
    if retrieved_top1:
        meta_r = corpus_meta_dict.get(retrieved_top1, ('?', '?', 0))
        print(f'  Récupéré  : {meta_r[0]} p.{meta_r[1]} (ocr_len={meta_r[2]})')

    worst_analysis.append({
        'query_id': qid, 'ndcg_a': score,
        'query': query_text,
        'expected_doc': rel_info[0] if rel_info else 'N/A'
    })

df_worst_a = pd.DataFrame(worst_analysis)
print('\n\nRésumé tableau des 10 pires requêtes A :')
display(df_worst_a[['query', 'ndcg_a', 'expected_doc']])

TOP 10 PIRES REQUÊTES — Approche A (textuelle)

nDCG@5=0.000 | QID: 1
  Requête   : FDA meeting response times before and during pandemic surge
  Attendu   : NASEUAworkshop21_PCavazzoni_forposting p.4 (score=2, ocr_len=514)
  Récupéré  : State_of_CDER_FDLI2021_PCavazzoni_20210515 p.1 (ocr_len=32)

nDCG@5=0.000 | QID: 6
  Requête   : Identify the FDA programs focused on enhancing access to generic medications and explain how these initiatives connect with actions taken to address the opioid epidemic.
  Attendu   : AAMGRxBiosimsSept2018_UHL p.26 (score=1, ocr_len=766) | AAMGRxBiosimsSept2018_UHL p.37 (score=1, ocr_len=482) | AAMGRxBiosimsSept2018_UHL p.40 (score=1, ocr_len=533) | AAMGRxBiosimsSept2018_UHL p.46 (score=1, ocr_len=438) | AAMGRxBiosimsSept2018_UHL p.6 (score=1, ocr_len=495) | State_of_CDER_FDLI2021_PCavazzoni_20210515 p.19 (score=1, ocr_len=568)
  Récupéré  : Presentation_FDA_Perspective_on_Abuse_Deterrent_Opioid_Development p.19 (ocr_len=421)

nDCG@5=0.000 | QID: 7
  Requêt

,query,ndcg_a,expected_doc
0,FDA meeting response times before and during p...,0.0,NASEUAworkshop21_PCavazzoni_forposting p.4 (sc...
1,Identify the FDA programs focused on enhancing...,0.0,"AAMGRxBiosimsSept2018_UHL p.26 (score=1, ocr_l..."
2,Contrast Anne-Emanuelle Birn's critique of str...,0.0,"drug_resistance_book p.318 (score=2, ocr_len=3..."
3,What distinguishes the approaches of proactive...,0.0,DDI_Medication_Errors_CE_March_2020_DMEPA_and_...
4,Explain how the tiered structure of evidence i...,0.0,Biosimilar_Regulatory_Policy_Understanding_the...
5,How do past performance data influence the dev...,0.0,FDA_Office_of_Generic_Drugs_OGD_Keynote_Addres...
6,Analyze the role of international regulatory c...,0.0,Future_of_Drug_Development_with_Janet_Woodcock...
7,patterns in SEND usage growth for investigatio...,0.0,"CDER_SEND_Webinar_June_2020 p.35 (score=1, ocr..."
8,"According to FDA guidelines, what is the requi...",0.0,DDI_Medication_Errors_CE_March_2020_DMEPA_and_...
9,How do the concentration-related risks of remd...,0.0,DDI_Medication_Errors_CE_March_2020_DMEPA_and_...


In [12]:
cache = {
    cid: emb.tolist()
    for cid, emb in zip(corpus_ids, corpus_embeddings_a)
}
with open("embeddings_textual_corpus.json", "w") as f:
    json.dump(cache, f)

print(f"Sauvegardé : embeddings_textual_corpus.json ({len(cache)} entrées)")

Sauvegardé : embeddings_textual_corpus.json (2313 entrées)


In [13]:
cache = {
    cid: emb.tolist()
    for cid, emb in zip(query_ids, query_embeddings_a)
}
with open("embeddings_textual_queries.json", "w") as f:
    json.dump(cache, f)

print(f"Sauvegardé : embeddings_textual_queries.json ({len(cache)} entrées)")

Sauvegardé : embeddings_textual_queries.json (2184 entrées)
